[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch08-model-training/03-bo-nho-huan-luyen.ipynb)

# Bộ nhớ huấn luyện: bốn số hạng, độ chính xác hỗn hợp và lưu điểm

Suy luận chỉ cần trọng số. Huấn luyện thì phải giữ thêm gradient, trạng thái bộ tối ưu (optimizer
state), và mọi giá trị kích hoạt (activation) mà lượt ngược sẽ cần. Chương 8 viết điều đó thành
phương trình bộ nhớ huấn luyện với bốn số hạng, rồi dùng nó để giải thích vì sao GPT-2 XL không vừa
một V100. Sổ tay này đếm từng số hạng bằng byte, trên tensor thật, rồi đi theo hai đòn bẩy của
chương: độ chính xác hỗn hợp và lưu điểm gradient (gradient checkpointing).

**Bạn sẽ làm:**
1. đếm bốn số hạng trên một mạng nhỏ, xem số hạng nào lớn lên theo batch, rồi nhân công thức 16 byte
   mỗi tham số lên mô hình 7 tỷ, 20 tỷ và 175 tỷ tham số;
2. xem autograd giữ lại đúng những tensor nào, đếm hệ số 5 byte của ma trận điểm attention, rồi dựng
   lại bảng giá trị kích hoạt từng tầng của GPT-2;
3. đọc ba định dạng số FP32, FP16 và BF16 từ chính PyTorch, xem gradient thật rơi khỏi miền của FP16,
   và xem co giãn mất mát (loss scaling) kéo chúng về;
4. đặt lưu điểm cách $\sqrt{N}$ tầng, đếm bộ nhớ và số lần tính lại của `torch.utils.checkpoint`, rồi
   đi lại phía bộ nhớ của ca GPT-2 trên một V100.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

## 1. Bốn số hạng, đếm bằng byte

Phương trình bộ nhớ huấn luyện của chương có bốn số hạng: trọng số, gradient, trạng thái bộ tối ưu và
giá trị kích hoạt. Ba số hạng đầu chỉ phụ thuộc mô hình. Số hạng cuối thì nhân theo batch, vì nó là
những gì lượt xuôi phải để lại cho lượt ngược.

PyTorch cho đếm số hạng cuối một cách chính xác. `torch.autograd.graph.saved_tensors_hooks` gọi một
hàm của bạn mỗi khi autograd **lưu** một tensor để dùng trong lượt ngược. Hàm `dem_kich_hoat` dưới đây
cộng byte của mọi tensor được lưu, mỗi vùng nhớ một lần, và bỏ qua chính tham số, vì tham số đã có số
hạng riêng.

**Dự đoán:** mạng 64-512-512-10 trên tập chữ số viết tay `load_digits` (ảnh 8×8), huấn luyện bằng
Adam. Ở batch nào thì giá trị kích hoạt vượt ba số hạng kia cộng lại: 16, 256, hay phải gần tới cả
tập dữ liệu?

In [ ]:
from sklearn.datasets import load_digits

nn = torch.nn
F = torch.nn.functional

Xd, yd = load_digits(return_X_y=True)
Xd = torch.tensor(Xd / 16.0, dtype=torch.float32)
yd = torch.tensor(yd)


def byte_cua(t):
    return t.numel() * t.element_size()


def dem_kich_hoat(mo_hinh, ham):
    """Chạy ham() và cộng byte mọi tensor autograd lưu cho lượt ngược, mỗi vùng nhớ một lần, trừ tham số."""
    tham_so = {p.data_ptr() for p in mo_hinh.parameters()} if mo_hinh is not None else set()
    giu = {}

    def goi(t):
        if t.data_ptr() not in tham_so:
            giu[t.data_ptr()] = max(giu.get(t.data_ptr(), 0), byte_cua(t))
        return t

    with torch.autograd.graph.saved_tensors_hooks(goi, lambda t: t):
        ket_qua = ham()
    return sum(giu.values()), ket_qua


def mlp_digits():
    dat_hat_giong()
    return nn.Sequential(nn.Linear(64, 512), nn.ReLU(), nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 10))


def so_cai(b):
    m = mlp_digits()
    bo = torch.optim.Adam(m.parameters(), lr=1e-3)
    kich_hoat, mat_mat = dem_kich_hoat(m, lambda: F.cross_entropy(m(Xd[:b]), yd[:b]))
    mat_mat.backward()
    bo.step()
    return {
        "trọng số": sum(byte_cua(p) for p in m.parameters()),
        "gradient": sum(byte_cua(p.grad) for p in m.parameters()),
        "trạng thái Adam": sum(byte_cua(v) for st in bo.state.values() for v in st.values()
                               if torch.is_tensor(v) and v.dim() > 0),   # bỏ bộ đếm bước, một số vô hướng
        "giá trị kích hoạt": kich_hoat,
    }


cac_batch = [16, 64, 256, 1024, len(Xd)]
bang = {b: so_cai(b) for b in cac_batch}
print(f"{'batch':>6}" + "".join(f"{k:>19}" for k in bang[16]) + "   (KB)")
for b, hang in bang.items():
    print(f"{b:>6}" + "".join(f"{v / 1e3:>19,.1f}" for v in hang.values()))

hang = bang[16]
assert hang["gradient"] == hang["trọng số"]              # mỗi tham số một gradient, cùng kiểu
assert hang["trạng thái Adam"] == 2 * hang["trọng số"]   # hai mô-men, mỗi cái cùng hình dạng tham số
kh = [bang[b]["giá trị kích hoạt"] for b in cac_batch]
moi_mau = {(kh[i + 1] - kh[i]) / (cac_batch[i + 1] - cac_batch[i]) for i in range(len(cac_batch) - 1)}
assert len(moi_mau) == 1                                  # đúng tuyến tính theo batch
print(f"mỗi mẫu thêm đúng {moi_mau.pop():,.0f} byte giá trị kích hoạt; ba số hạng kia không đổi theo batch")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
day = np.zeros(len(cac_batch))
for ten, mau in zip(bang[16], ("C0", "C1", "C2", "C3")):
    cot = np.array([bang[b][ten] / 1e6 for b in cac_batch])
    ax.bar(range(len(cac_batch)), cot, bottom=day, color=mau, label=ten, width=0.6)
    day += cot
tinh_mb = (hang["trọng số"] + hang["gradient"] + hang["trạng thái Adam"]) / 1e6
ax.axhline(2 * tinh_mb, color="0.4", ls=":", lw=1)
ax.annotate("kích hoạt = ba số hạng tĩnh", (-0.3, 2 * tinh_mb), xytext=(0, 4), textcoords="offset points",
            fontsize=8, color="0.3")
ax.set_xticks(range(len(cac_batch)), [str(b) for b in cac_batch])
ax.set_xlabel("kích thước batch")
ax.set_ylabel("MB")
ax.set_ylim(0, day.max() * 1.15)
ax.set_title("Bốn số hạng của mạng 64-512-512-10 với Adam (đếm)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Mạng này hẹp, khoảng 300 nghìn tham số, nên phần tĩnh chiếm ưu thế tới tận batch rất lớn. Ba số hạng
tĩnh đúng như chương nói: gradient bằng đúng trọng số, và Adam giữ thêm hai lần trọng số. Với
transformer thì ngược lại: mỗi mẫu là cả một chuỗi token, mỗi token để lại một vector cho mỗi tầng, nên
giá trị kích hoạt vượt phần tĩnh ngay ở batch 4, như phần 2 sẽ cho thấy.

### Công thức 16 byte mỗi tham số

Ở độ chính xác hỗn hợp, công thức của chương giữ trọng số và gradient ở FP16, cộng một bản trọng số chủ
(master weights) FP32 mà bộ tối ưu cập nhật, và hai mô-men Adam ở FP32. Số byte của mỗi kiểu lấy thẳng
từ PyTorch:

In [ ]:
B16 = torch.empty(0, dtype=torch.float16).element_size()
B32 = torch.empty(0, dtype=torch.float32).element_size()
cong_thuc = {"trọng số FP16": B16, "gradient FP16": B16, "trọng số chủ FP32": B32, "hai mô-men Adam FP32": 2 * B32}
print(" + ".join(f"{v} ({k})" for k, v in cong_thuc.items()))
BYTE_MOI_THAM_SO = sum(cong_thuc.values())
theo_sach("byte mỗi tham số", BYTE_MOI_THAM_SO, sach=16, nguon="16 byte mỗi tham số")
theo_sach("huấn luyện so với suy luận chỉ giữ trọng số FP16", BYTE_MOI_THAM_SO / B16, sach=8, nguon="8 lần · 7 tỷ")


def ngan_sach(n_tham_so):
    """GB của từng khoản trong công thức, trước giá trị kích hoạt."""
    return {k: n_tham_so * v / 1e9 for k, v in cong_thuc.items()}


bay_ty = ngan_sach(sach(7e9, nguon="8 lần · 7 tỷ"))
print("7 tỷ tham số:", " + ".join(f"{v:,.0f}" for v in bay_ty.values()), "GB")
theo_sach("trọng số FP16 (GB)", bay_ty["trọng số FP16"], sach=14, nguon="14+14+28+56 = 112 GB")
theo_sach("tổng trước giá trị kích hoạt (GB)", sum(bay_ty.values()), sach=112, nguon="14+14+28+56 = 112 GB")
print()

Napkin Math 1.4 của chương hỏi: mô hình 7 tỷ tham số có huấn luyện được trên một GPU 24 GB không? Phần
giá trị kích hoạt dùng ước lượng đơn giản hoá cho transformer: batch nhân độ dài chuỗi (sequence
length), nhân chiều ẩn, nhân số tầng, nhân số byte, nhân một hệ số cho các giá trị trung gian được giữ
lại.

In [ ]:
kh_7b = (sach(1, trich="With Batch = 1, Seq = 2048, Hidden = 4096, 32 layers") * sach(2048, nguon="2048 · 4096 · 32 tầng")
         * sach(4096, nguon="2048 · 4096 · 32 tầng") * sach(32, nguon="2048 · 4096 · 32 tầng")
         * B16 * sach(57, nguon="hệ số ≈57", tol=0.5)) / 1e9
theo_sach("giá trị kích hoạt, batch 1 (GB)", kh_7b, sach=30.6, nguon="30,6 GB · hệ số ≈57", tol=0.05)
GPU_24 = sach(24, trich="fit on a 24 GB GPU for training")
print(f"cộng lại: {sum(bay_ty.values()):,.0f} + {kh_7b:.1f} = {sum(bay_ty.values()) + kh_7b:,.1f} GB, so với {GPU_24} GB")
print(f"riêng phần trước giá trị kích hoạt đã gấp {sum(bay_ty.values()) / GPU_24:.1f} lần dung lượng GPU")

Câu trả lời là không, và không cần tới số hạng giá trị kích hoạt để biết. Cùng công thức ấy cho mô hình
20 tỷ tham số trong phần ngộ nhận của chương, và cho GPT-3 khi chỉ đếm tham số:

In [ ]:
hai_muoi_ty = ngan_sach(sach(20e9, nguon="20 tỷ · 40+40+80+160 = 320 GB"))
print("20 tỷ tham số:", " + ".join(f"{v:,.0f}" for v in hai_muoi_ty.values()), "GB")
theo_sach("hai mô-men Adam (GB)", hai_muoi_ty["hai mô-men Adam FP32"], sach=160, nguon="40+40+80+160 = 320 GB · 160 GB")
theo_sach("trạng thái phi kích hoạt (GB)", sum(hai_muoi_ty.values()), sach=320, nguon="40+40+80+160 = 320 GB")
N_GPT3 = sach(175e9, nguon="175 tỷ")
theo_sach("GPT-3, chỉ tham số FP32 (GB)", N_GPT3 * B32 / 1e9, sach=700, nguon="≈700 GB FP32", tol=0.5)
theo_sach("GPT-3, chỉ tham số FP16 (GB)", N_GPT3 * B16 / 1e9, sach=350, nguon="350 GB FP16")
print(f"GPT-3 theo công thức 16 byte: {N_GPT3 * BYTE_MOI_THAM_SO / 1e12:.1f} TB, trước giá trị kích hoạt")

### ResNet-50 ở batch 32

Chương lấy ResNet-50 xử lý ảnh 224×224. Hình dạng đầu ra của tầng tích chập đầu tiên thì không cần tin
sách: dựng kiến trúc của `torchvision` (chỉ kiến trúc, không tải trọng số nào) và chạy một ảnh qua nó.

In [ ]:
import torchvision

dat_hat_giong()
resnet = torchvision.models.resnet50(weights=None)
hinh = {}
resnet.conv1.register_forward_hook(lambda m, i, o: hinh.__setitem__("conv1", tuple(o.shape[1:])))
with torch.no_grad():
    resnet(torch.zeros(1, 3, 224, 224))
KENH, CAO, RONG_ANH = hinh["conv1"]
print(f"đầu ra tầng tích chập đầu: {KENH} kênh × {CAO} × {RONG_ANH}")
theo_sach("chiều cao bản đồ", CAO, sach=112, nguon="3,2 MB · 112×112×64")
theo_sach("số kênh", KENH, sach=64, nguon="3,2 MB · 112×112×64")
theo_sach("một ảnh ở FP32 (MB)", KENH * CAO * RONG_ANH * B32 / 1e6, sach=3.2, nguon="3,2 MB · 112×112×64")

THAM_SO_MB = sach(102.4, nguon="102,4 MB")
n_resnet = sum(p.numel() for p in resnet.parameters())
print(f"tham số đếm được: {n_resnet:,} × {B32} byte = {n_resnet * B32 / 1e6:.1f} MB; sách: {THAM_SO_MB} MB,"
      f" tức {THAM_SO_MB / B32:.1f} triệu tham số")

KH_XUOI = sach(8, nguon="8 GB + 4 GB + 102,4 MB = 12,1 GB")
KH_NGUOC = sach(4, nguon="8 GB + 4 GB + 102,4 MB = 12,1 GB")
A100 = sach(80, nguon="15,1 phần trăm · 80 GB")
tong32 = KH_XUOI + KH_NGUOC + THAM_SO_MB / 1e3
theo_sach("batch 32 (GB)", tong32, sach=12.1, nguon="= 12,1 GB")
theo_sach("phần của một A100 80 GB (%)", 100 * tong32 / A100, sach=15.1, nguon="15,1 phần trăm")
theo_sach("batch 64 (GB)", 2 * KH_XUOI + 2 * KH_NGUOC + THAM_SO_MB / 1e3, sach=24.1, nguon="16 GB + 8 GB = 24,1 GB")
print("(8 GB và 4 GB nhân đôi theo batch, 102.4 MB tham số thì không)")

102.4 MB ứng đúng 25.6 triệu tham số nhân 4 byte, tức sách làm tròn số tham số lên một chữ số thập
phân. Còn 8 GB và 4 GB là tổng của cả 50 tầng mà chương đưa ra, không phải thứ hình dạng một tầng suy
ra được. Hai số ấy nhân đôi khi batch nhân đôi, còn tham số thì đứng yên: đúng điều phần đầu vừa đếm.

## 2. Autograd giữ lại những gì

Chương nói lượt xuôi tính và lưu cả tổng tuyến tính trước hàm kích hoạt lẫn giá trị kích hoạt sau nó,
và tập được giữ chính xác tuỳ vào phép toán. Ô dưới đếm, với khối `Linear → hàm kích hoạt → Linear` ở
batch 32 và độ rộng 256, autograd giữ bao nhiêu tensor cỡ 32×256.

**Dự đoán:** ReLU và GELU có giữ cùng số tensor không?

In [ ]:
giu_theo_ham = {}
for ham_kh in (nn.ReLU(), nn.Sigmoid(), nn.Tanh(), nn.GELU()):
    khoi_nho = nn.Sequential(nn.Linear(256, 256), ham_kh, nn.Linear(256, 256))
    x = torch.randn(32, 256)
    so_byte, _ = dem_kich_hoat(khoi_nho, lambda: khoi_nho(x))
    giu_theo_ham[type(ham_kh).__name__] = so_byte // byte_cua(x)
    print(f"{type(ham_kh).__name__:<8} giữ {so_byte // byte_cua(x)} tensor cỡ 32×256")
assert giu_theo_ham == {"ReLU": 2, "Sigmoid": 2, "Tanh": 2, "GELU": 3}

mnist = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 10))
so_byte, _ = dem_kich_hoat(mnist, lambda: F.cross_entropy(mnist(torch.zeros(1, 784)), torch.zeros(1, dtype=torch.long)))
print(f"\nmạng ba tầng 784-128-64-10, một ảnh MNIST: {so_byte:,} byte giá trị kích hoạt")

ReLU, sigmoid và tanh tính được đạo hàm từ chính đầu ra của chúng, nên chỉ đầu ra được giữ, và nó cũng
là đầu vào mà `Linear` sau cần: một vùng nhớ, đếm một lần. GELU cần đầu vào của nó, nên khối GELU giữ cả
tổng tuyến tính lẫn giá trị kích hoạt. Mạng ba tầng cho MNIST thì cần cỡ vài kilobyte mỗi ảnh, đúng như
chương nói. Khoảng cách từ đó tới GPT-2 là chủ đề của phần còn lại.

### Ma trận điểm attention: 5 byte mỗi phần tử

Napkin Math 1.3 của chương tách giá trị kích hoạt của một tầng GPT-2 thành bốn khoản. Ba khoản tuyến
tính theo độ dài chuỗi $S$. Khoản thứ tư, ma trận điểm attention, có $S^2$ phần tử cho mỗi đầu, và sách
tính nó là $5 \times N_{đầu} \times S^2 \times B$ byte cho các bộ đệm của điểm, softmax và dropout. Vì
sao là 5?

Ô dưới chạy một attention thật ở BF16 (16 bit, cùng số byte với FP16), có dropout, rồi tách các tensor
được lưu thành hai nhóm: hình $S \times S$, và phần còn lại. Dropout ở đây gọi `torch.native_dropout`,
kernel hợp nhất mà `F.dropout` dùng trên GPU. Nó giữ mặt nạ ở kiểu `bool`, 1 byte mỗi phần tử.

**Dự đoán:** mỗi phần tử $S \times S$ được lưu bao nhiêu byte? Con số ấy có đổi khi $S$ tăng không?

In [ ]:
def attention(q, k, v, p=0.1):
    diem = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
    xac_suat = torch.softmax(diem, dim=-1)
    xac_suat, _ = torch.native_dropout(xac_suat, p, True)
    return xac_suat @ v


B_NHO, DAU_NHO, D_DAU = 2, 4, 16
byte_moi_o = {}
print(f"{'S':>5} {'byte mỗi phần tử S×S':>22} {'byte mỗi phần tử của Q':>24}")
for S_ in (64, 128, 256, 512):
    dat_hat_giong()
    q, k, v = (torch.randn(B_NHO, DAU_NHO, S_, D_DAU, dtype=torch.bfloat16, requires_grad=True) for _ in range(3))
    giu = {}

    def goi(t):
        giu[t.data_ptr()] = (tuple(t.shape[-2:]), byte_cua(t))
        return t

    with torch.autograd.graph.saved_tensors_hooks(goi, lambda t: t):
        attention(q, k, v)
    vuong = sum(b for h, b in giu.values() if h == (S_, S_))
    con_lai = sum(b for h, b in giu.values() if h != (S_, S_))
    byte_moi_o[S_] = vuong / (B_NHO * DAU_NHO * S_ * S_)
    print(f"{S_:>5} {byte_moi_o[S_]:>22.0f} {con_lai / (B_NHO * DAU_NHO * S_ * D_DAU):>24.0f}")
assert set(byte_moi_o.values()) == {5}
BYTE_DIEM = byte_moi_o[64]
theo_sach("hệ số byte của ma trận điểm", BYTE_DIEM, sach=5, trich="\\(5 \\times N_{\\text{heads}}")
print()

Năm byte là softmax giữ đầu ra của nó (2 byte), dropout giữ mặt nạ (1 byte), và phép nhân với $V$ giữ
ma trận sau dropout (2 byte). Điểm thô trước softmax thì không được giữ, vì đạo hàm của softmax chỉ cần
đầu ra. Phần còn lại là $Q$, $K$, $V$, mỗi cái 2 byte, và nó chỉ tuyến tính theo $S$.

Bây giờ dựng lại bảng của chương, với $B = 4$, $S = 1{,}024$, $d = 1{,}600$, 25 đầu và 48 tầng ở 16 bit.

**Dự đoán:** ở $S = 1{,}024$, ma trận điểm chiếm bao nhiêu phần của một tầng? Khi $S$ gấp đôi, nó gấp
mấy lần?

In [ ]:
B = sach(4, nguon="B=4 · S=1024 · d=1600 · 48 tầng")
S = sach(1024, nguon="B=4 · S=1024 · d=1600 · 48 tầng")
D = sach(1600, nguon="B=4 · S=1024 · d=1600 · 48 tầng")
N_TANG = sach(48, nguon="B=4 · S=1024 · d=1600 · 48 tầng")
N_DAU = sach(25, nguon="524,3 MB mỗi tầng · 25 đầu")
CHUAN_HOA = sach(10, nguon="≈10 MB mỗi tầng", tol=0.5)


def mot_tang(s, b=B):
    """MB giá trị kích hoạt của ba khoản có công thức trong Napkin Math 1.3, ở 16 bit."""
    return {
        "attention (Q, K, V, đầu ra)": b * s * D * 4 * B16 / 1e6,
        "truyền thẳng (phần nở 4d)": b * s * 4 * D * B16 / 1e6,
        "ma trận điểm (S×S)": BYTE_DIEM * N_DAU * s**2 * b / 1e6,
    }


tang = dict(mot_tang(S), **{"chuẩn hoá (sách ước lượng)": CHUAN_HOA})
for ten, mb in tang.items():
    print(f"{ten:<28} {mb:8.1f} MB")
theo_sach("attention (MB)", tang["attention (Q, K, V, đầu ra)"], sach=52.4, nguon="52,4 MB · 52,4 MB")
theo_sach("truyền thẳng (MB)", tang["truyền thẳng (phần nở 4d)"], sach=52.4, nguon="52,4 MB · 52,4 MB")
theo_sach("ma trận điểm (MB)", tang["ma trận điểm (S×S)"], sach=524.3, nguon="524,3 MB mỗi tầng")
TONG_TANG = sum(tang.values())
theo_sach("tổng một tầng (MB)", TONG_TANG, sach=639.1, nguon="≈639,1 MB mỗi tầng", tol=0.05)
print(f"ma trận điểm chiếm {100 * tang['ma trận điểm (S×S)'] / TONG_TANG:.0f} % của một tầng")

In [ ]:
cac_s = np.array([128, 256, 512, 1024, 2048, 4096, 8192])
fig, ax = plt.subplots(figsize=(8, 4.2))
for ten, mau, kieu, co in zip(mot_tang(S), ("C0", "C2", "C3"), ("o-", "s--", "^-"), (10, 4, 6)):
    ax.loglog(cac_s, [mot_tang(s)[ten] for s in cac_s], kieu, color=mau, ms=co, mfc="none" if co == 10 else mau,
              lw=2.5 if co == 10 else 1.5, label=ten)
ax.annotate(f"{tang['ma trận điểm (S×S)']:.1f} MB", (S, tang["ma trận điểm (S×S)"]), xytext=(-62, 8),
            textcoords="offset points", fontsize=8, color="C3")
ax.annotate(f"{tang['attention (Q, K, V, đầu ra)']:.1f} MB (cả hai)", (S, tang["attention (Q, K, V, đầu ra)"]),
            xytext=(6, -14), textcoords="offset points", fontsize=8, color="C0")
ax.axvline(S, color="0.6", ls=":", lw=1)
ax.set_xticks(cac_s, [f"{s:,}" for s in cac_s])
ax.xaxis.set_minor_locator(plt.NullLocator())
ax.set_xlabel("độ dài chuỗi S")
ax.set_ylabel("MB mỗi tầng (B = 4, 16 bit)")
ax.set_title("Một tầng GPT-2 XL: ma trận điểm bậc hai, phần còn lại bậc một")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Hai đường tuyến tính trùng nhau, vì $4 \times 2$ byte của attention bằng đúng phần nở $4d$ nhân 2 byte
của mạng truyền thẳng. Ma trận điểm dốc gấp đôi trên thang log: gấp đôi $S$ là gấp bốn nó.

Cộng 48 tầng lại:

In [ ]:
cong_48 = N_TANG * TONG_TANG / 1e3
THU_VIEN = sach(35.9, nguon="35,9 GB")
print(f"48 tầng × {TONG_TANG:.1f} MB = {cong_48:.1f} GB")
print(f"ước lượng của thư viện mà sách dùng:  {THU_VIEN} GB")

N_XL = sach(1.5e9, nguon="1,5 tỷ (XL)")
tinh = {"trọng số FP16": N_XL * B16 / 1e9, "gradient FP16": N_XL * B16 / 1e9,
        "trọng số chủ FP32": N_XL * B32 / 1e9, "hai mô-men Adam FP32": N_XL * 2 * B32 / 1e9}
print("trạng thái tĩnh:", " + ".join(f"{v:g}" for v in tinh.values()), "GB")
theo_sach("trọng số FP16 (GB)", tinh["trọng số FP16"], sach=3, nguon="3 + 3 + 6 + 12 GB")
theo_sach("trọng số chủ FP32 (GB)", tinh["trọng số chủ FP32"], sach=6, nguon="3 + 3 + 6 + 12 GB")
theo_sach("hai mô-men Adam (GB)", tinh["hai mô-men Adam FP32"], sach=12, nguon="3 + 3 + 6 + 12 GB")
TINH = sum(tinh.values())
theo_sach("đỉnh khi huấn luyện (GB)", THU_VIEN + TINH, sach=59.9, nguon="≈59,9 GB", tol=0.05)
print(f"giá trị kích hoạt chiếm {100 * THU_VIEN / (THU_VIEN + TINH):.0f} % của đỉnh")

Bốn khoản của bảng cộng lại chỉ được khoảng 30.7 GB, ít hơn 35.9 GB. Đây không phải lỗi cộng: sách ghi
rõ 35.9 GB là ước lượng của thư viện, gồm cả dòng dư (residual stream) và các bộ đệm của framework,
những thứ nằm ngoài bốn khoản đã liệt kê. Sổ tay giữ cả hai con số và dùng 35.9 GB từ đây, như chương.

Đỉnh 59.9 GB thì vượt dung lượng 32 GB của V100. Đòn bẩy đầu tiên chương đưa ra là **tính lại chọn
lọc**: bỏ ma trận điểm của mọi tầng rồi tính lại nó trong lượt ngược.

In [ ]:
V100 = sach(32, nguon="6 + 6 + 12 = 24 GB · 32 GB")
bo_diem = N_TANG * tang["ma trận điểm (S×S)"] / 1e3
sau_khi_bo = THU_VIEN - bo_diem
theo_sach("giá trị kích hoạt sau tính lại chọn lọc (GB)", sau_khi_bo, sach=10.7, nguon="≈10,7 GB", tol=0.05)
theo_sach("cộng trạng thái tĩnh (GB)", sau_khi_bo + TINH, sach=34.7, nguon="≈34,7 GB", tol=0.05)
GIAM_CHON_LOC = sach(70.2, nguon="70,2 phần trăm")
print(f"mức giảm: tính được {100 * bo_diem / THU_VIEN:.1f} %, sách ghi {GIAM_CHON_LOC} %")
print(f"{sau_khi_bo + TINH:.1f} GB vẫn trên {V100} GB của V100, trước cả vùng làm việc tạm và phụ trội bộ cấp phát")

10.7 GB và 34.7 GB khớp, còn tỉ lệ giảm lệch 0.1 điểm phần trăm. Chỗ lệch ấy không nằm ở phép trừ mà ở
chỗ 35.9 GB đã được làm tròn. Phần 4 sẽ gặp thêm ba chỗ lệch cùng cỡ, và một lời giải thích chung cho
cả bốn.

## 3. Ba định dạng số

Bảng 10 của chương so FP32, FP16 và BF16. `torch.finfo` cho mọi con số của bảng ấy từ chính định dạng:
số bit, `eps` (khoảng cách từ 1 tới số biểu diễn được kế tiếp, nên số bit định trị là
$-\log_2 \mathrm{eps}$), giá trị lớn nhất và giá trị chuẩn nhỏ nhất (`tiny`). Số chữ số thập phân là
$\log_{10} 2^{p}$, với $p$ là số bit định trị cộng một bit ẩn.

**Dự đoán:** FP16 và BF16 cùng 16 bit. Định dạng nào giữ được 70,000? Định dạng nào phân biệt được 1
với 1.001?

In [ ]:
dinh_dang = {"FP32": torch.float32, "FP16": torch.float16, "BF16": torch.bfloat16}
so = {}
for ten, kieu in dinh_dang.items():
    fi = torch.finfo(kieu)
    dinh_tri = round(-math.log2(fi.eps))
    so[ten] = {"mu": fi.bits - 1 - dinh_tri, "dinh_tri": dinh_tri, "tiny": fi.tiny,
               "chu_so": (dinh_tri + 1) * math.log10(2)}
    print(f"{ten}: {fi.bits} bit = 1 dấu + {so[ten]['mu']} số mũ + {dinh_tri} định trị · lớn nhất {fi.max:.5g}"
          f" · chuẩn nhỏ nhất {fi.tiny:.3g} · {so[ten]['chu_so']:.2f} chữ số thập phân")

theo_sach("FP32: bit số mũ", so["FP32"]["mu"], sach=8, nguon="8 · 5 · 8 bit số mũ")
theo_sach("FP16: bit số mũ", so["FP16"]["mu"], sach=5, nguon="8 · 5 · 8 bit số mũ")
theo_sach("BF16: bit số mũ", so["BF16"]["mu"], sach=8, nguon="8 · 5 · 8 bit số mũ")
theo_sach("FP32: bit định trị", so["FP32"]["dinh_tri"], sach=23, nguon="23 · 10 · 7 bit định trị")
theo_sach("FP16: bit định trị", so["FP16"]["dinh_tri"], sach=10, nguon="23 · 10 · 7 bit định trị")
theo_sach("BF16: bit định trị", so["BF16"]["dinh_tri"], sach=7, nguon="23 · 10 · 7 bit định trị")

theo_sach("FP16: giá trị lớn nhất", torch.finfo(torch.float16).max, sach=65_504, nguon="6,1·10⁻⁵ · 65.504")
theo_sach("FP16: giá trị chuẩn nhỏ nhất", so["FP16"]["tiny"], sach=6.1e-5, nguon="6,1·10⁻⁵ · 65.504")
duoi_chuan_nho_nhat = torch.tensor(1, dtype=torch.int16).view(torch.float16).item()   # mẫu bit 0…01
theo_sach("FP16: số dưới chuẩn nhỏ nhất", duoi_chuan_nho_nhat, sach=6e-8, nguon="≈6·10⁻⁸", tol=5e-9)
for ten in ("FP32", "BF16"):
    bac = float(f"1e{math.floor(math.log10(so[ten]['tiny']))}")
    theo_sach(f"{ten}: bậc của giá trị chuẩn nhỏ nhất", bac, sach=1e-38, nguon="10⁻³⁸")

theo_sach("FP32: chữ số thập phân", so["FP32"]["chu_so"], sach=7, nguon="≈7 · ≈3", tol=0.5)
theo_sach("FP16: chữ số thập phân", so["FP16"]["chu_so"], sach=3, nguon="≈7 · ≈3", tol=0.5)
thap = sach(2, nguon="≈2 đến 3 chữ số thập phân", tol=0.5)
cao = sach(3, nguon="≈2 đến 3 chữ số thập phân", tol=0.5)
print(f"BF16: {so['BF16']['chu_so']:.2f} chữ số thập phân, trong khoảng {thap} đến {cao} của sách")
assert thap <= so["BF16"]["chu_so"] <= cao

print()
for ten, kieu in dinh_dang.items():
    print(f"{ten}: 70,000 → {torch.tensor(70_000.0).to(kieu).item():,.0f} · 1.001 → {torch.tensor(1.001).to(kieu).item():.7f}")

BF16 giữ 8 bit số mũ của FP32 nên giữ được 70,000 (làm tròn thành 70,144) và mọi thứ tới cỡ
$10^{-38}$. Đổi lại, 7 bit định trị không phân biệt được 1 với 1.001. FP16 thì ngược lại: phân biệt
được 1.001, nhưng tràn trên (overflow) thành vô cực ở 70,000, và miền chuẩn của nó dừng ở
$6.1 \times 10^{-5}$. Đó là lựa chọn mà chương gọi là ưu tiên miền động hơn độ chính xác.

### Gradient thật rơi khỏi miền FP16

Dưới $6.1 \times 10^{-5}$, FP16 chỉ còn số dưới chuẩn (subnormal), mất dần chữ số, rồi dưới cỡ
$6 \times 10^{-8}$ thì tràn dưới (underflow) về 0. Ô dưới huấn luyện một mạng bốn tầng trên
`load_digits` ở FP32 cho tới khi mất mát gần 0, lấy gradient trên cả tập, rồi ép chúng sang FP16: ép
thẳng, và ép sau khi nhân với $2^{10}$, hệ số ví dụ của chương. Co giãn mất mát nhân **mất mát** với hệ
số ấy, và vì đạo hàm tuyến tính nên mọi gradient được nhân đúng hệ số ấy. Ở đây ta nhân thẳng gradient
FP32, để phép so sánh chỉ phụ thuộc phép làm tròn của FP16.

**Dự đoán:** khi ép thẳng, phần lớn gradient nằm trong miền chuẩn, trong miền dưới chuẩn, hay bằng 0?

In [ ]:
dat_hat_giong()
bon_tang = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(),
                         nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 10))
bo = torch.optim.Adam(bon_tang.parameters(), lr=1e-3)
for epoch in range(30):
    for idx in torch.randperm(len(Xd)).split(64):
        bo.zero_grad()
        F.cross_entropy(bon_tang(Xd[idx]), yd[idx]).backward()
        bo.step()
bo.zero_grad()
mat_mat = F.cross_entropy(bon_tang(Xd), yd)
mat_mat.backward()
g = torch.cat([p.grad.flatten() for p in bon_tang.parameters()])
g = g[g != 0]                       # bỏ những gradient vốn bằng đúng 0, như đơn vị ReLU đã tắt
print(f"mất mát trên cả tập: {mat_mat.item():.4f} · {g.numel():,} gradient khác 0 ở FP32")

HE_SO = sach(1024, nguon="2¹⁰")
TINY16 = torch.finfo(torch.float16).tiny


def ep_fp16(he_so):
    g16 = (g * he_so).half()
    ve_0 = (g16 == 0).float().mean().item()
    duoi_chuan = ((g16 != 0) & (g16.abs() < TINY16)).float().mean().item()
    sai = ((g16.float() / he_so - g).abs() / g.abs()).median().item()
    return ve_0, duoi_chuan, sai


for nhan, he_so in (("ép thẳng", 1), ("nhân 2¹⁰", HE_SO)):
    ve_0, duoi_chuan, sai = ep_fp16(he_so)
    print(f"FP16, {nhan:<9} về 0 {100 * ve_0:5.2f} % · dưới chuẩn {100 * duoi_chuan:5.1f} %"
          f" · miền chuẩn {100 * (1 - ve_0 - duoi_chuan):5.1f} % · sai số tương đối trung vị sau khi chia lại {sai:.1e}")
gb = g.bfloat16()
print(f"BF16, ép thẳng  về 0 {100 * (gb == 0).float().mean().item():5.2f} %"
      f" · sai số tương đối trung vị {((gb.float() - g).abs() / g.abs()).median().item():.1e}")

In [ ]:
log2g = torch.log2(g.abs()).numpy()
fig, ax = plt.subplots(figsize=(8.5, 3.8))
khung = np.arange(np.floor(log2g.min()) - 1, np.ceil(log2g.max()) + 12, 0.5)
ax.hist(log2g, bins=khung, color="0.6", label="gradient FP32")
ax.hist(log2g + 10, bins=khung, histtype="step", color="C0", lw=1.5, label="sau khi nhân 2¹⁰")
ax.set_ylim(0, ax.get_ylim()[1] * 1.6)
ax.axvspan(khung[0], -24, color="C3", alpha=0.12)
ax.axvspan(-24, -14, color="C1", alpha=0.12)
ax.axvline(-24, color="C3", lw=1)
ax.axvline(-14, color="C1", lw=1)
dinh_y = ax.get_ylim()[1]
ax.text(-25, dinh_y * 0.95, "về 0 ở FP16\ndưới 2⁻²⁴ ≈ 6·10⁻⁸", ha="right", va="top", fontsize=8, color="C3")
ax.text(-19, dinh_y * 0.95, "dưới chuẩn", ha="center", va="top", fontsize=8, color="C1")
ax.text(-13, dinh_y * 0.95, "miền chuẩn\ntừ 2⁻¹⁴ ≈ 6.1·10⁻⁵", ha="left", va="top", fontsize=8, color="0.3")
ax.set_xlabel("log₂ |gradient|")
ax.set_ylabel("số gradient")
ax.set_title("Gradient của một mạng đã học gần xong, so với miền của FP16")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Ép thẳng, phần lớn gradient rơi vào miền dưới chuẩn, nơi chúng mất dần bit định trị, và một phần về
đúng 0: những cập nhật ấy mất hẳn. Nhân $2^{10}$ dịch cả phân bố sang phải 10 bậc nhị phân, nên phần lớn
vào miền chuẩn, và sau khi chia lại ở FP32 thì sai số nhỏ hơn hẳn. BF16 không cần bước này vì miền số mũ
của nó rộng như FP32, nhưng mỗi gradient chỉ còn khoảng 2 tới 3 chữ số. Bộ co giãn động trong thực tế
(như `torch.amp.GradScaler`) tự hạ hệ số khi gặp vô cực hoặc NaN, và tự nâng nó lên khi một thời gian
không gặp.

### Trọng số chủ FP32, và chính sách theo từng phép

Một bước cập nhật thường nhỏ hơn trọng số nhiều bậc. Gần 1, các số FP16 cách nhau `eps` của nó.

**Dự đoán:** cộng 1,000 lần một bước 0.0001 vào trọng số 1.0. Trọng số FP16 thành bao nhiêu?

In [ ]:
BUOC = 1e-4
w = {ten: torch.tensor(1.0, dtype=kieu) for ten, kieu in dinh_dang.items()}
for _ in range(1000):
    for ten in w:
        w[ten] += BUOC
for ten, kieu in dinh_dang.items():
    print(f"{ten}: {w[ten].item():.6f} · eps = {torch.finfo(kieu).eps:.2e}")
assert w["FP16"].item() == 1.0 and w["BF16"].item() == 1.0 and abs(w["FP32"].item() - 1.1) < 1e-3

lop = nn.Linear(64, 32)
with torch.autocast("cpu", dtype=torch.bfloat16):
    dau_ra = lop(torch.randn(8, 64))
    mat_mat_ac = F.cross_entropy(dau_ra, torch.zeros(8, dtype=torch.long))
mat_mat_ac.backward()
print(f"\nautocast: trọng số {lop.weight.dtype} · đầu ra Linear {dau_ra.dtype} · mất mát {mat_mat_ac.dtype}"
      f" · gradient của trọng số {lop.weight.grad.dtype}")
assert lop.weight.dtype == torch.float32 and dau_ra.dtype == torch.bfloat16

N_TY = sach(1e9, nguon="10⁹ tham số → 4 GB xuống 2 GB")
theo_sach("trọng số FP32 của 10⁹ tham số (GB)", N_TY * B32 / 1e9, sach=4, nguon="10⁹ tham số → 4 GB xuống 2 GB")
theo_sach("ở FP16 hoặc BF16 (GB)", N_TY * torch.empty(0, dtype=torch.bfloat16).element_size() / 1e9, sach=2,
          nguon="4 GB xuống 2 GB")
theo_sach("mức giảm bộ nhớ trọng số (%)", 100 * (1 - B16 / B32), sach=50, nguon="50 phần trăm")
print()

Bước 0.0001 nhỏ hơn nửa `eps` của FP16 và BF16, nên mỗi phép cộng làm tròn về đúng 1.0. Một nghìn
bước cập nhật biến mất mà không có lỗi nào được báo. Bản trọng số chủ FP32 cộng dồn chúng đúng. Đó là
lý do công thức của chương trả thêm 4 byte mỗi tham số.

`torch.autocast` cho thấy điều chương gọi là khế ước số học chia đôi. Trọng số vẫn là FP32, phép nhân
ma trận chạy ở BF16, còn hàm mất mát được đưa về FP32. Danh sách phép nào chạy ở độ chính xác nào là
chính sách của framework, và mỗi loại thiết bị có danh sách riêng.

Mức giảm 50 phần trăm chỉ là của tensor trọng số. Tổng bộ nhớ giảm ít hơn, vì công thức vẫn giữ trọng
số chủ và trạng thái bộ tối ưu ở FP32.

## 4. Lưu điểm cách $\sqrt{N}$ tầng

Bảng 15 của chương xét một mạng $N_L$ tầng, mỗi tầng lưu $A$ byte. Không lưu điểm thì bộ nhớ là
$N_L \cdot A$. Với $k$ lưu điểm, lượt xuôi chỉ giữ $k$ ranh giới. Lượt ngược tính lại từng đoạn
$N_L/k$ tầng khi tới nó, nên đỉnh là $k \cdot A + (N_L/k) \cdot A$, đổi lấy $N_L - k$ phép xuôi tính
lại. Đạo hàm theo $k$ bằng 0 ở $k = \sqrt{N_L}$.

**Dự đoán:** với $N_L = 48$ như GPT-2, đỉnh ở $k$ tối ưu bằng bao nhiêu phần của $48A$?

In [ ]:
N_L = N_TANG
cac_k = np.arange(1, N_L + 1)
dinh = cac_k + N_L / cac_k
k_tot = int(cac_k[np.argmin(dinh)])
print(f"√{N_L} = {math.sqrt(N_L):.2f}; k nguyên tốt nhất là {k_tot}, đỉnh {dinh[k_tot - 1]:.2f}·A")
theo_sach("số lưu điểm", k_tot, sach=7, trich="\\(7 \\times A + (48/7) \\times A\\)")
theo_sach("đỉnh (đơn vị A)", dinh[k_tot - 1], sach=14, trich="approximately equals \\(14 \\times A\\)", tol=0.5)
theo_sach("tiết kiệm bộ nhớ (%)", 100 * (1 - dinh[k_tot - 1] / N_L), sach=71, nguon="71 phần trăm")

XUOI_NGUOC = sach(3, nguon="3 lần")   # lượt xuôi cộng ngược so với riêng lượt xuôi
theo_sach("phụ trội nếu tính lại trọn một lượt xuôi (%)", 100 / XUOI_NGUOC, sach=33, nguon="≈33 phần trăm", tol=0.5)
print(f"theo cách đếm của Bảng 15, {N_L} − {k_tot} = {N_L - k_tot} tầng tính lại: "
      f"{100 * (N_L - k_tot) / N_L / XUOI_NGUOC:.1f} % thêm vào một bước")

Hai con số phụ trội không mâu thuẫn. 33 phần trăm là mô hình đơn giản của chương: tính lại cả một lượt
xuôi, trong một bước vốn tốn 3 lượt xuôi. Bảng 15 thì đếm chặt hơn, $N_L - k$ tầng, vì tầng cuối của mỗi
đoạn đã có đầu ra là lưu điểm kế tiếp. Còn PyTorch làm theo cách nào?

### Đếm trên `torch.utils.checkpoint`

Ô dưới dựng một mạng 48 khối `Linear → ReLU`, chia thành $k$ đoạn gần bằng nhau, và bọc mỗi đoạn trong
`checkpoint(..., use_reentrant=False)`. Hai thứ được đếm, không đo: byte autograd còn giữ khi lượt xuôi
xong (cùng hook ở phần 1, tính theo $A$ là byte một đầu ra khối), và số lần các lớp `Linear` chạy lượt
xuôi **trong** lượt ngược, tức số khối được tính lại.

**Dự đoán:** ở $k = 7$, PyTorch tính lại 41 khối như Bảng 15, hay khác?

In [ ]:
from torch.utils.checkpoint import checkpoint

RONG, LO = 256, 256
dat_hat_giong()
khoi = nn.ModuleList([nn.Sequential(nn.Linear(RONG, RONG), nn.ReLU()) for _ in range(N_L)])
dem_xuoi = [0]
for kh_ in khoi:
    kh_[0].register_forward_hook(lambda m, i, o: dem_xuoi.__setitem__(0, dem_xuoi[0] + 1))


def ranh_gioi(k):
    return [round(i * N_L / k) for i in range(k + 1)]


def chay(x, k):
    if k == 0:
        for kh_ in khoi:
            x = kh_(x)
        return x
    rg = ranh_gioi(k)
    for a, b in zip(rg[:-1], rg[1:]):
        def doan(h, a=a, b=b):
            for kh_ in khoi[a:b]:
                h = kh_(h)
            return h
        x = checkpoint(doan, x, use_reentrant=False)
    return x


A = LO * RONG * B32
cac_k_do = [0, 1, 2, 3, 4, 6, 7, 8, 12, 16, 24, 48]
dem = {}
print(f"{'k':>3} {'giữ sau lượt xuôi':>18} {'đoạn dài nhất':>14} {'đỉnh':>6} {'k + N/k':>8} {'tính lại':>9} {'N − k':>6}")
for k in cac_k_do:
    x = torch.randn(LO, RONG, requires_grad=True)
    dem_xuoi[0] = 0
    giu_lai, ra = dem_kich_hoat(khoi, lambda: chay(x, k).sum())
    xuoi = dem_xuoi[0]
    ra.backward()
    tinh_lai = dem_xuoi[0] - xuoi
    dai_nhat = max(b - a for a, b in zip(ranh_gioi(k)[:-1], ranh_gioi(k)[1:])) if k else 0
    dem[k] = (giu_lai / A, dai_nhat, tinh_lai)
    print(f"{k:>3} {giu_lai / A:>16.0f}·A {dai_nhat:>14} {giu_lai / A + dai_nhat:>5.0f}A"
          f" {(k + N_L / k if k else N_L):>8.1f} {tinh_lai:>9} {(N_L - k if k else 0):>6}")
    assert xuoi == N_L
    assert giu_lai == (k if k else N_L + 1) * A
    assert tinh_lai == (N_L if k else 0)

In [ ]:
def mot_buoc(k):
    x = torch.randn(LO, RONG, requires_grad=True)
    chay(x, k).sum().backward()


thoi_gian = {}
for k in (0, 7):
    mot_buoc(k)
    lan = []
    for _ in range(5):
        t0 = time.perf_counter()
        mot_buoc(k)
        lan.append(time.perf_counter() - t0)
    thoi_gian[k] = min(lan)
    do_tren_may(f"một bước xuôi cộng ngược, k = {k}", 1e3 * thoi_gian[k], "ms")
do_tren_may("thời gian thêm của k = 7", 100 * (thoi_gian[7] / thoi_gian[0] - 1), "%")

In [ ]:
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [1.5, 1]})
ax.plot(cac_k, dinh, color="0.5", lw=1.2, label="Bảng 15: k·A + (N/k)·A")
cac_k_ck = [k for k in cac_k_do if k]
ax.plot(cac_k_ck, [dem[k][0] + dem[k][1] for k in cac_k_ck], "o", color="C3",
        label="đếm: giữ sau lượt xuôi + đoạn dài nhất")
ax.plot(cac_k_ck, [dem[k][0] for k in cac_k_ck], "s", color="C0", ms=4, label="đếm: giữ sau lượt xuôi")
ax.axhline(dem[0][0], color="C1", ls="--", lw=1)
ax.annotate(f"không lưu điểm: {dem[0][0]:.0f}·A", (24, dem[0][0]), xytext=(0, -12), textcoords="offset points",
            fontsize=8, color="C1", ha="center")
ax.annotate(f"k = {k_tot}: {dem[k_tot][0] + dem[k_tot][1]:.0f}·A", (k_tot, dem[k_tot][0] + dem[k_tot][1]),
            xytext=(4, 28), textcoords="offset points", fontsize=8, color="C3",
            arrowprops=dict(arrowstyle="-", color="C3", lw=0.8))
ax.set_xscale("log")
ax.set_xticks(cac_k_ck, [str(k) for k in cac_k_ck])
ax.xaxis.set_minor_locator(plt.NullLocator())
ax.set_ylim(0, 60)
ax.set_xlabel("số lưu điểm k (thang log)")
ax.set_ylabel("bộ nhớ giá trị kích hoạt, đơn vị A")
ax.set_title(f"Đỉnh bộ nhớ, N = {N_L} khối")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=2, fontsize=8, frameon=False)
ax2.plot(cac_k_ck, [N_L - k for k in cac_k_ck], color="0.5", lw=1.2, label="Bảng 15: N − k")
ax2.plot(cac_k_ck, [dem[k][2] for k in cac_k_ck], "o", color="C3", label="PyTorch (đếm)")
ax2.set_xscale("log")
ax2.set_xticks(cac_k_ck, [str(k) for k in cac_k_ck])
ax2.xaxis.set_minor_locator(plt.NullLocator())
ax2.set_ylim(0, 55)
ax2.set_xlabel("số lưu điểm k (thang log)")
ax2.set_ylabel("khối được tính lại")
ax2.set_title("Số khối tính lại trong lượt ngược")
ax2.legend(loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=2, fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Phần bộ nhớ đúng như Bảng 15. Sau lượt xuôi, PyTorch chỉ giữ đầu vào của $k$ đoạn. Khi lượt ngược tới
một đoạn, đoạn ấy được tính lại và giữ trọn, nên đỉnh là $k$ cộng độ dài đoạn dài nhất: thấp nhất là
14·A, ở $k$ từ 6 tới 8. (Không lưu điểm thì là 49 chứ không phải 48, vì đầu vào của mạng cũng được giữ.)

Phần tính lại thì khác. PyTorch chạy lại **cả** đoạn từ đầu vào của nó, kể cả khối cuối, vì ReLU của
khối cuối cần chính đầu ra của nó trong lượt ngược, và lưu điểm không biết tensor ấy trùng với đầu vào
của đoạn sau. Nên nó tính lại đủ 48 khối với mọi $k$, đúng mô hình "thêm một lượt xuôi" ứng với
khoảng 33 phần trăm. Ở $k = 48$, mỗi khối một đoạn, nó vừa giữ 48·A vừa tính lại cả 48 khối: tệ cả hai
đầu.

### GPT-2 trên một V100: phía bộ nhớ của ca nghiên cứu

Ca nghiên cứu của chương lập nền ở FP32 với batch 4, rồi áp hai can thiệp lên bộ nhớ: độ chính xác hỗn
hợp, và lưu điểm gradient với **giả định minh hoạ** rằng giá trị kích hoạt giảm bốn lần.

In [ ]:
KH_FP32 = sach(71.7, nguon="71,7 GB")
GIAM_LUU_DIEM = sach(4, nguon="3 đến 4 lần · 35,9 GB xuống 9 GB")
cac_buoc = {
    "nền FP32": {"trọng số": N_XL * B32 / 1e9, "gradient": N_XL * B32 / 1e9, "trọng số chủ": 0.0,
                 "mô-men Adam": N_XL * 2 * B32 / 1e9, "giá trị kích hoạt": KH_FP32},
    "độ chính xác hỗn hợp": {"trọng số": tinh["trọng số FP16"], "gradient": tinh["gradient FP16"],
                             "trọng số chủ": tinh["trọng số chủ FP32"], "mô-men Adam": tinh["hai mô-men Adam FP32"],
                             "giá trị kích hoạt": THU_VIEN},
}
cac_buoc["+ lưu điểm"] = dict(cac_buoc["độ chính xác hỗn hợp"], **{"giá trị kích hoạt": THU_VIEN / GIAM_LUU_DIEM})
tong = {ten: sum(b.values()) for ten, b in cac_buoc.items()}
for ten, t in tong.items():
    print(f"{ten:<22} " + " + ".join(f"{v:.1f}" for v in cac_buoc[ten].values()) + f" = {t:.1f} GB")
theo_sach("nền FP32 (GB)", tong["nền FP32"], sach=95.7, nguon="95,7 GB → 33 GB")
theo_sach("sau độ chính xác hỗn hợp (GB)", tong["độ chính xác hỗn hợp"], sach=59.9, nguon="59,9 GB · 37,5 phần trăm")
theo_sach("giá trị kích hoạt sau lưu điểm (GB)", cac_buoc["+ lưu điểm"]["giá trị kích hoạt"], sach=9,
          nguon="35,9 GB xuống 9 GB")
theo_sach("sau lưu điểm (GB)", tong["+ lưu điểm"], sach=33, nguon="95,7 GB → 33 GB")
theo_sach("mức giảm tổng", tong["nền FP32"] / tong["+ lưu điểm"], sach=2.9, nguon="95,7 GB → 33 GB · 2,9 lần")
GIAM_HON_HOP = sach(37.5, nguon="59,9 GB · 37,5 phần trăm")
print(f"mức giảm của riêng độ chính xác hỗn hợp: tính được "
      f"{100 * (1 - tong['độ chính xác hỗn hợp'] / tong['nền FP32']):.1f} %, sách ghi {GIAM_HON_HOP} %")
print(f"FP32 từ 35.9 GB: 2 × {THU_VIEN} = {2 * THU_VIEN:.1f} GB, sách ghi {KH_FP32} GB")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
day = np.zeros(len(cac_buoc))
for muc, mau in zip(cac_buoc["nền FP32"], ("C0", "C1", "C4", "C2", "C3")):
    cot = np.array([b[muc] for b in cac_buoc.values()])
    ax.bar(range(len(cac_buoc)), cot, bottom=day, color=mau, label=muc, width=0.55)
    day += cot
for i, t in enumerate(tong.values()):
    ax.annotate(f"{t:.1f} GB", (i, t), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)
ax.axhline(V100, color="k", ls="--", lw=1)
ax.annotate(f"V100: {V100} GB", (2.4, V100), xytext=(0, -11), textcoords="offset points", fontsize=8, ha="left")
ax.set_xticks(range(len(cac_buoc)), list(cac_buoc))
ax.set_xlim(-0.5, 3.0)
ax.set_ylim(0, 108)
ax.set_ylabel("GB (mô hình tensor của chương, batch 4)")
ax.set_title("GPT-2 XL trên một V100: hai can thiệp lên bộ nhớ")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Độ chính xác hỗn hợp đổi số hạng đầu, lưu điểm đổi số hạng giá trị kích hoạt, và tổng 33 GB vẫn nhích
trên 32 GB trước cả vùng làm việc tạm. Mức giảm 35.9 GB xuống 9 GB chính là câu "3 đến 4 lần" ở phần
tổng kết của chương.

### Batch 32, và bốn chỗ lệch 0.1 GB

Bảng tổng hợp ở cuối phần tối ưu của chương làm lại phép tính ở batch 32, tức gấp 8 lần batch 4.

In [ ]:
GAP_BATCH = sach(32, trich="Activations (batch=32)") / B
KH32_FP32 = sach(573.8, nguon="573,8 → 71,7 GB")
KH32_TOT = sach(71.7, nguon="573,8 → 71,7 GB")
tinh_fp32 = tong["nền FP32"] - KH_FP32          # 6 + 6 + 12 GB, không đổi theo batch
tong32_fp32, tong32_tot = tinh_fp32 + KH32_FP32, TINH + KH32_TOT
theo_sach("tổng FP32, batch 32 (GB)", tong32_fp32, sach=597.8, nguon="597,8 GB → 95,7 GB")
theo_sach("tổng tối ưu, batch 32 (GB)", tong32_tot, sach=95.7, nguon="597,8 GB → 95,7 GB")
theo_sach("tỉ số", tong32_fp32 / tong32_tot, sach=6, nguon="597,8 GB → 95,7 GB · 6 lần")
print(f"giá trị kích hoạt FP32 từ batch 4: {GAP_BATCH:.0f} × {KH_FP32} = {GAP_BATCH * KH_FP32:.1f} GB, sách ghi {KH32_FP32} GB")
print(f"giá trị kích hoạt tối ưu từ batch 4: {GAP_BATCH:.0f} × {THU_VIEN} / {GIAM_LUU_DIEM}"
      f" = {GAP_BATCH * THU_VIEN / GIAM_LUU_DIEM:.1f} GB, sách ghi {KH32_TOT} GB")

Bốn phép tính của chương lệch đúng một chữ số cuối khi làm lại từ các số đã in: 70.1 so với 70.2 phần
trăm, 71.8 so với 71.7 GB, 37.4 so với 37.5 phần trăm, 573.6 so với 573.8 GB (và 71.8 so với 71.7 GB ở
batch 32 tối ưu, cùng phép nhân đôi ấy). Cả bốn đều đi qua 35.9 GB, hoặc qua bản nhân đôi 71.7 GB của nó.
Giả thuyết của sổ tay này: chương tính từ một tổng chưa làm tròn, và 35.9 là bản in của nó. Ô dưới tìm
khoảng tổng mà 573.8 GB cho phép, rồi thử một giá trị trong khoảng ấy với mọi con số của ca nghiên cứu.
Đây là **giả định** của sổ tay, không phải số của sách.

In [ ]:
CHAT = 0.05    # nửa đơn vị của chữ số cuối mà sách in
thap_x = (KH32_FP32 - CHAT) / (2 * GAP_BATCH)
cao_x = (KH32_FP32 + CHAT) / (2 * GAP_BATCH)
x = 35.86      # giả định: một tổng chưa làm tròn trong khoảng trên
print(f"573.8 GB = 2 (FP32 so với 16 bit) × {GAP_BATCH:.0f} (batch) × tổng → tổng trong [{thap_x:.3f}, {cao_x:.3f}) GB;"
      f" thử x = {x}")
thu = {
    "tổng giá trị kích hoạt (GB)": (x, THU_VIEN),
    "đỉnh khi huấn luyện (GB)": (x + TINH, sach(59.9, nguon="≈59,9 GB", tol=0.05)),
    "tính lại chọn lọc (%)": (100 * bo_diem / x, GIAM_CHON_LOC),
    "giá trị kích hoạt FP32 (GB)": (2 * x, KH_FP32),
    "nền FP32 (GB)": (2 * x + TINH, sach(95.7, nguon="95,7 GB → 33 GB")),
    "mức giảm của độ chính xác hỗn hợp (%)": (100 * x / (2 * x + TINH), GIAM_HON_HOP),
    "giá trị kích hoạt FP32, batch 32 (GB)": (2 * GAP_BATCH * x, KH32_FP32),
    "giá trị kích hoạt tối ưu, batch 32 (GB)": (GAP_BATCH * x / GIAM_LUU_DIEM, KH32_TOT),
}
for ten, (tinh_x, in_sach) in thu.items():
    khop = round(tinh_x, 1) == in_sach
    print(f"{'✓' if khop else '✗'} {ten:<40} {tinh_x:9.3f} → {round(tinh_x, 1):7.1f} · sách {in_sach}")
    assert khop

Một tổng duy nhất, quanh 35.86 GB, làm tròn ra đúng mọi con số của ca nghiên cứu. Bài học không nằm ở
0.1 GB, mà ở thói quen: khi làm lại một chuỗi phép tính từ những số đã làm tròn, chữ số cuối sẽ trôi.
Hãy kiểm phép tính bằng sai số làm tròn của nó, đừng kiểm bằng dấu bằng.

## Thử thêm

1. Ở phần 1, thay Adam bằng `torch.optim.SGD(m.parameters(), lr=0.1, momentum=0.9)`. Số hạng nào
   co lại, và co mấy lần? Batch nào thì giá trị kích hoạt vượt phần tĩnh?
2. Ở ô `scores`, thay `torch.native_dropout(xac_suat, p, True)` bằng `F.dropout(xac_suat, p, True), None`.
   Trên CPU, hệ số thành bao nhiêu byte, và vì sao? (Gợi ý: xem kiểu của tensor $S \times S$ được giữ.)
3. Ở phần 4, đổi `N_L = N_TANG` thành `N_L = 100`. $k$ tối ưu có đúng là 10 không, đỉnh còn bao nhiêu
   phần của $100A$, và PyTorch tính lại bao nhiêu khối?

## Tóm lại

* Huấn luyện giữ bốn số hạng: trọng số, gradient bằng đúng trọng số, trạng thái Adam gấp đôi trọng số,
  và giá trị kích hoạt tuyến tính theo batch. Công thức độ chính xác hỗn hợp của chương là 16 byte mỗi
  tham số, gấp 8 lần suy luận, nên mô hình 7 tỷ cần 112 GB trước khi đếm giá trị kích hoạt.
* Autograd giữ đúng những gì đạo hàm cần. Ở attention, đó là 5 byte cho mỗi phần tử của ma trận điểm,
  một số hạng bậc hai theo độ dài chuỗi, chiếm phần lớn giá trị kích hoạt của GPT-2.
* FP16 có 10 bit định trị nhưng miền chuẩn dừng ở $6.1 \times 10^{-5}$, nên gradient nhỏ rơi vào miền
  dưới chuẩn hoặc về 0 nếu không co giãn mất mát. BF16 giữ miền số mũ của FP32, đổi lại chỉ còn 2 tới
  3 chữ số. Trọng số chủ FP32 giữ những bước cập nhật nhỏ hơn nửa `eps`.
* Lưu điểm cách $\sqrt{N}$ tầng hạ đỉnh từ $N \cdot A$ xuống khoảng $2\sqrt{N} \cdot A$, tức 71 phần
  trăm ở 48 tầng. PyTorch tính lại trọn mỗi đoạn, nên cái giá là cả một lượt xuôi thêm.
* Với GPT-2 XL batch 4, hai đòn bẩy đưa 95.7 GB xuống 33 GB, vẫn nhích trên một V100 32 GB.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Huấn luyện mô hình — khi định luật sắt trở thành công cụ hằng ngày](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch08-model-training/), dựng từ chương
[*Model Training*](https://mlsysbook.ai/vol1/training/training.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_08_model_train.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.